# NARMAX — Ball-and-Beam

Applies the **NARX/FROLS** nonlinear system identification approach
(mirrored from `narmax_example_drone.ipynb`) to the real ball-and-beam
dataset used in A2, with the same experiment and pre-processing settings.

| Signal | Physical meaning | Unit | Range |
|---|---|---|---|
| `u` | Beam tilt angle (servo command) | degrees | −5 to +8 |
| `y` | Ball position along beam | cm | −4.5 to +4 |

Sampling: **Ts = 50 ms** (20 Hz) after 50× decimation from 1 kHz raw data.

**A2 values mirrored:** `EXPERIMENT`, `RESAMPLE_FACTOR`, `TRAIN_FRACTION`, `NY=N_A=2`, `NU=N_B=2`

Requires:
```
pip install git+https://github.com/helonayala/bab_datasets.git
```

## Imports, Functions and Classes

In [ ]:
import numpy as np
from itertools import combinations_with_replacement
import matplotlib.pyplot as plt
import bab_datasets as nod

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 5)


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def evaluate(y_true, y_pred):
    residuals = y_true - y_pred
    rmse = np.sqrt(np.mean(residuals ** 2))
    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = 1.0 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return rmse, r2


# ---------------------------------------------------------------------------
# Regression matrix builders
# ---------------------------------------------------------------------------
def regMatARX(y_signal_in, u_signal_in, ny, nu):
    y = np.asarray(y_signal_in, dtype=float)
    u = np.asarray(u_signal_in, dtype=float)
    max_lag = max(ny, nu)
    y_target = y[max_lag:]
    colnames = [f'y(k-{i})' for i in range(1, ny + 1)] + \
               [f'u(k-{i})' for i in range(1, nu + 1)]
    rows = []
    for k in range(max_lag, len(y)):
        rows.append([y[k - j] for j in range(1, ny + 1)] +
                    [u[k - j] for j in range(1, nu + 1)])
    return np.array(rows, dtype=float), colnames, y_target


def regMatNARX(u_signal_in, y_signal_in, nu, ny, poly_order):
    y = np.asarray(y_signal_in, dtype=float)
    u = np.asarray(u_signal_in, dtype=float)
    P0, P0_names, y_target = regMatARX(y, u, ny, nu)
    NP, n_base = len(y_target), P0.shape[1]
    cols, names = [np.ones((NP, 1))], ['constant']
    cols.append(P0)
    names.extend(P0_names)
    if poly_order >= 2:
        for order in range(2, poly_order + 1):
            for idx_t in combinations_with_replacement(range(n_base), order):
                names.append(''.join(P0_names[i] for i in idx_t))
                cols.append(np.prod(P0[:, list(idx_t)], axis=1, keepdims=True))
    return np.concatenate(cols, axis=1), names, y_target


# ---------------------------------------------------------------------------
# FROLS
# ---------------------------------------------------------------------------
def frols_py(P, Y_in, n_terms, colnames=None, eps=1e-12):
    Y = Y_in.reshape(-1, 1)
    NP, M = P.shape
    sig_yy = (Y.T @ Y).item() or eps
    sel_idx, err_list, g_list = [], [], []
    Q = np.empty((NP, 0))
    A = np.empty((0, 0))

    for _ in range(min(n_terms, M)):
        best_err, best_i, best_q, best_g = -np.inf, -1, None, 0.0
        for m in range(M):
            if m in sel_idx:
                continue
            p_m = P[:, m:m+1]
            q = p_m.copy()
            for r in range(Q.shape[1]):
                q_r = Q[:, r:r+1]
                denom = (q_r.T @ q_r).item()
                if denom >= eps:
                    q -= ((p_m.T @ q_r).item() / denom) * q_r
            q_norm = (q.T @ q).item()
            if q_norm < eps:
                continue
            g = (Y.T @ q).item() / q_norm
            err = g ** 2 * q_norm / sig_yy
            if err > best_err:
                best_err, best_i, best_q, best_g = err, m, q, g
        if best_i == -1:
            break
        sel_idx.append(best_i)
        err_list.append(best_err)
        g_list.append(best_g)
        Q = best_q if Q.shape[1] == 0 else np.hstack((Q, best_q))
        p_new = P[:, best_i:best_i+1]
        if A.size == 0:
            A = np.array([[1.0]])
        else:
            col = np.zeros((A.shape[0], 1))
            for r in range(Q.shape[1] - 1):
                q_r = Q[:, r:r+1]
                denom = (q_r.T @ q_r).item()
                if denom >= eps:
                    col[r, 0] = (p_new.T @ q_r).item() / denom
            A = np.block([[A, col],
                          [np.zeros((1, A.shape[1])), np.ones((1, 1))]])
    if not sel_idx:
        return {}
    theta = np.linalg.solve(A, np.array(g_list).reshape(-1, 1)).flatten()
    return {'th': theta, 'selected_indices': sel_idx,
            'Psel_colnames': [colnames[i] for i in sel_idx] if colnames else [],
            'ERR_values': np.array(err_list)}


# ---------------------------------------------------------------------------
# NARX class
# ---------------------------------------------------------------------------
class NARX:
    def __init__(self, ny, nu, poly_order, n_terms):
        self.ny, self.nu = ny, nu
        self.poly_order = poly_order
        self.n_terms = n_terms
        self._max_lag = max(ny, nu)
        self._P0_names = ([f'y(k-{i})' for i in range(1, ny + 1)] +
                          [f'u(k-{i})' for i in range(1, nu + 1)])
        self._n_base = len(self._P0_names)
        self.theta_ = self.selected_indices_ = None
        self.selected_colnames_ = self.candidate_colnames_ = self.err_values_ = None

    def fit(self, u, y):
        P, names, y_t = regMatNARX(u, y, self.nu, self.ny, self.poly_order)
        self.candidate_colnames_ = names
        res = frols_py(P, y_t, self.n_terms, names)
        self.theta_             = res['th']
        self.selected_indices_  = res['selected_indices']
        self.selected_colnames_ = res['Psel_colnames']
        self.err_values_        = res['ERR_values']
        return self

    def predict_osa(self, u, y):
        P, _, y_t = regMatNARX(u, y, self.nu, self.ny, self.poly_order)
        return P[:, self.selected_indices_] @ self.theta_, y_t

    def simulate(self, u, y_init):
        u = np.asarray(u, dtype=float)
        N = len(u)
        y_hat = np.zeros(N)
        y_hat[:self._max_lag] = np.asarray(y_init[:self._max_lag], dtype=float)

        def _row(y_buf, k):
            P0_vals = [y_buf[k - j] for j in range(1, self.ny + 1)] + \
                      [u[k - j]     for j in range(1, self.nu + 1)]
            terms = {'constant': 1.0}
            for i, nm in enumerate(self._P0_names):
                terms[nm] = P0_vals[i]
            if self.poly_order >= 2:
                for order in range(2, self.poly_order + 1):
                    for idx_t in combinations_with_replacement(range(self._n_base), order):
                        nm = ''.join(self._P0_names[i] for i in idx_t)
                        terms[nm] = np.prod([P0_vals[i] for i in idx_t])
            return np.array([terms[c] for c in self.candidate_colnames_])

        for k in range(self._max_lag, N):
            y_hat[k] = _row(y_hat, k)[self.selected_indices_] @ self.theta_
        return y_hat[self._max_lag:]


# ---------------------------------------------------------------------------
# Data loader
# ---------------------------------------------------------------------------
def load_data(experiment='multisine_05', resample_factor=50, y_dot_method='savgol'):
    data = nod.load_experiment(
        experiment, preprocess=True, plot=False, end_idx=None,
        resample_factor=resample_factor, zoom_last_n=10000,
        y_dot_method=y_dot_method,
    )
    return (np.asarray(data.u, dtype=float).ravel(),
            np.asarray(data.y, dtype=float).ravel(),
            float(data.sampling_time))

print('All functions and classes loaded.')

## Configuration

In [ ]:
# --- Data settings (mirrored from A2) ---
EXPERIMENT      = 'multisine_05'
RESAMPLE_FACTOR = 50          # Ts = 50 ms  (20 Hz)
TRAIN_FRACTION  = 0.5

# --- NARX hyper-parameters (best from grid search) ---
NY          = 2   # output lags   ← mirrors A2 N_A = 2
NU          = 2   # input lags    ← mirrors A2 N_B = 2
POLY_ORDER  = 3   # polynomial expansion order
NO_OF_TERMS = 10  # terms FROLS selects

## Load and Visualise Data

In [ ]:
u, y, ts = load_data(EXPERIMENT, RESAMPLE_FACTOR)
t = np.arange(len(u)) * ts

fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
axes[0].plot(t, y, 'tab:blue', lw=0.8, label='ball position')
axes[0].set_ylabel('y  [cm]')
axes[0].legend(); axes[0].grid(True)

axes[1].plot(t, u, 'tab:red', lw=0.8, label='beam angle')
axes[1].set_ylabel('u  [deg]')
axes[1].set_xlabel('time [s]')
axes[1].legend(); axes[1].grid(True)

fig.suptitle(f'{EXPERIMENT}  —  Ts={ts*1000:.0f} ms  ({len(u)} samples after {RESAMPLE_FACTOR}× decimation)',
             fontsize=11)
plt.tight_layout()
plt.show()

print(f'u: min={u.min():.2f}  max={u.max():.2f}  mean={u.mean():.3f}  [deg]')
print(f'y: min={y.min():.2f}  max={y.max():.2f}  mean={y.mean():.3f}  [cm]')

## Train / Test Split

In [ ]:
n     = len(u)
split = int(round(TRAIN_FRACTION * n))

u_tra, y_tra = u[:split], y[:split]
u_tst, y_tst = u[split:], y[split:]

print(f'Total samples : {n}')
print(f'Train  (50 %) : {split}   ({split * ts:.1f} s)')
print(f'Test   (50 %) : {n - split}   ({(n - split) * ts:.1f} s)')

## Define Model Parameters and Fit

FROLS selects the `NO_OF_TERMS` most significant regressors from a pool of
candidate polynomial combinations of past outputs and inputs up to `POLY_ORDER`.

In [ ]:
model = NARX(ny=NY, nu=NU, poly_order=POLY_ORDER, n_terms=NO_OF_TERMS)
model.fit(u_tra, y_tra)

print(f'NARX(ny={NY}, nu={NU}, poly_order={POLY_ORDER}, n_terms={NO_OF_TERMS})')
print(f'Candidate pool : {len(model.candidate_colnames_)} terms')
print(f'Selected       : {len(model.selected_colnames_)} terms\n')
print(f'{"Term":<32} {"theta":>12}   {"ERR (%)":>10}')
print('-' * 60)
for nm, th, err in zip(model.selected_colnames_, model.theta_, model.err_values_):
    print(f'{nm:<32} {th:>+12.6f}   {err*100:>10.4f}%')
print('-' * 60)
print(f'{"Total ERR explained":<32} {"":>12}   {model.err_values_.sum()*100:>10.4f}%')

## One-Step-Ahead (OSA) Prediction

At each step the model uses the **true** past output values.
OSA R² ≈ 0.996 on both halves, confirming accurate parameter estimates
and no overfitting.

In [ ]:
yhat_tra_osa, Y_tra = model.predict_osa(u_tra, y_tra)
yhat_tst_osa, Y_tst = model.predict_osa(u_tst, y_tst)

rmse_tr, r2_tr = evaluate(Y_tra, yhat_tra_osa)
rmse_te, r2_te = evaluate(Y_tst, yhat_tst_osa)

print(f'{"":32s}  {"RMSE":>12}  {"R²":>8}')
print(f'{"OSA – train":32s}  {rmse_tr:>12.4e}  {r2_tr:>8.4f}')
print(f'{"OSA – test" :32s}  {rmse_te:>12.4e}  {r2_te:>8.4f}')

t_tra = np.arange(len(Y_tra)) * ts
t_tst = (split + np.arange(len(Y_tst))) * ts

fig, axes = plt.subplots(2, 1, figsize=(12, 8))
axes[0].plot(t_tra, Y_tra, 'k', lw=0.8, label='measured')
axes[0].plot(t_tra, yhat_tra_osa, '--', lw=1.0,
             label=f'NARX OSA  R²={r2_tr:.4f}')
axes[0].set_title(f'Training half — {EXPERIMENT}')
axes[0].set_ylabel('ball position [cm]')
axes[0].legend(fontsize=8); axes[0].grid(True)

axes[1].plot(t_tst, Y_tst, 'k', lw=0.8, label='measured')
axes[1].plot(t_tst, yhat_tst_osa, '--', lw=1.0,
             label=f'NARX OSA  R²={r2_te:.4f}')
axes[1].set_title('Test half')
axes[1].set_xlabel('time [s]')
axes[1].set_ylabel('ball position [cm]')
axes[1].legend(fontsize=8); axes[1].grid(True)

fig.suptitle(f'NARX OSA  (ny={NY}, nu={NU}, order={POLY_ORDER}, {NO_OF_TERMS} terms)',
             fontsize=11)
plt.tight_layout()
plt.show()

## Free-Run (FR) Simulation

The model runs autonomously: each predicted output is fed back as the next input.
Only the initial `max_lag` true values and the full input sequence `u` are provided.
FR R² is the honest test of dynamic model quality.

In [ ]:
max_lag    = model._max_lag
yhat_tr_fr = model.simulate(u_tra, y_tra[:max_lag])
Y_tra_fr   = y_tra[max_lag:]

rmse_fr, r2_fr = evaluate(Y_tra_fr, yhat_tr_fr)
print(f'{"FR – train":32s}  {rmse_fr:>12.4e}  {r2_fr:>8.4f}')

t_fr = np.arange(len(Y_tra_fr)) * ts

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(t_fr, Y_tra_fr,  'k',  lw=0.8, label='measured')
ax.plot(t_fr, yhat_tr_fr, '--', lw=1.0, label=f'NARX FR  R²={r2_fr:.4f}')
ax.set_title(f'Free-Run Simulation  (ny={NY}, nu={NU}, order={POLY_ORDER}, {NO_OF_TERMS} terms)')
ax.set_xlabel('time [s]')
ax.set_ylabel('ball position [cm]')
ax.legend(fontsize=8); ax.grid(True)
plt.tight_layout()
plt.show()

## Error Reduction Ratio (ERR)

ERR shows how much each selected term contributes to explaining output variance.
The first two linear AR terms (`y(k-1)`, `y(k-2)`) account for ≈ 97.6%,
confirming the system is near-linear with small nonlinear corrections.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = list(range(len(model.selected_colnames_)))
bars = ax.bar(x, model.err_values_ * 100, color='steelblue', edgecolor='k', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(model.selected_colnames_, rotation=30, ha='right', fontsize=9)
ax.set_ylabel('ERR (%)')
ax.set_title('Error Reduction Ratio per selected term')
for bar, val in zip(bars, model.err_values_ * 100):
    if val > 0.3:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.2,
                f'{val:.2f}%', ha='center', va='bottom', fontsize=8)
plt.tight_layout()
plt.show()

## Grid Search — Finding the Best Hyper-parameters

Sweep `NY ∈ [1–5]`, `NU ∈ [1–5]`, `POLY_ORDER ∈ [1, 2, 3]` with `n_terms=10` fixed.
Results are sorted by **FR R²** (descending).
Configurations that diverge numerically in free-run are flagged as `nan`.

In [ ]:
import warnings
import pandas as pd

NY_vals         = [1, 2, 3, 4, 5]
NU_vals         = [1, 2, 3, 4, 5]
POLY_ORDER_vals = [1, 2, 3]
N_TERMS         = 10

records = []
total = len(NY_vals) * len(NU_vals) * len(POLY_ORDER_vals)
done  = 0

for ny in NY_vals:
    for nu in NU_vals:
        for poly in POLY_ORDER_vals:
            done += 1
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter('ignore')
                    m = NARX(ny=ny, nu=nu, poly_order=poly, n_terms=N_TERMS)
                    m.fit(u_tra, y_tra)
                    yhat_tr, Y_tr = m.predict_osa(u_tra, y_tra)
                    yhat_te, Y_te = m.predict_osa(u_tst, y_tst)
                    _, r2_otr = evaluate(Y_tr, yhat_tr)
                    _, r2_ote = evaluate(Y_te, yhat_te)
                    yhat_fr  = m.simulate(u_tra, y_tra[:m._max_lag])
                    _, r2_fr = evaluate(y_tra[m._max_lag:], yhat_fr)
                records.append({'ny': ny, 'nu': nu, 'order': poly,
                                '#candidates': len(m.candidate_colnames_),
                                'R2_OSA_train': round(r2_otr, 4),
                                'R2_OSA_test':  round(r2_ote, 4),
                                'R2_FR_train':  round(r2_fr,  4) if np.isfinite(r2_fr) else float('nan')})
            except Exception as exc:
                records.append({'ny': ny, 'nu': nu, 'order': poly,
                                '#candidates': 0, 'R2_OSA_train': float('nan'),
                                'R2_OSA_test': float('nan'), 'R2_FR_train': float('nan')})
            print(f'\r  [{done}/{total}]', end='', flush=True)

print(f'\nDone.')

df = pd.DataFrame(records)
df_sorted = df.sort_values('R2_FR_train', ascending=False).reset_index(drop=True)
df_sorted.index += 1  # rank from 1

### Full results — sorted by FR R²

In [ ]:
pd.set_option('display.max_rows', 80)
pd.set_option('display.float_format', '{:.4f}'.format)
df_sorted

### Top 10 by FR R²

In [ ]:
top10 = df_sorted[df_sorted['R2_FR_train'].notna()].head(10)
top10

## Conclusions

| Finding | Evidence |
|---|---|
| System is **near-linear** | First 2 AR terms explain 97.6% ERR; linear FR R²≈0.74 |
| **POLY_ORDER=3** is essential for best FR | All top-10 use order=3 |
| **ny=2, nu=2** is the optimal lag structure | Matches 2nd-order mechanical system physics |
| OSA is insensitive to model quality | All 75 configs give OSA R²≈0.995–0.997 |
| Best model: `ny=2, nu=2, order=3, 10 terms` | FR R²=0.8815, OSA R²=0.9970 |